# 14 — ETL-pipelines for AI

**Fase:** 3 — Dataengineering | **Tid:** 2 timer | **Krav:** Notatbok 02

**Hva du bygger:** En ETL-pipeline som henter data fra en API, validerer og renser den, og laster den inn i en database — klar for RAG-ingestion.

---

## Hva er ETL?

```
Extract  → Hent rådata (API, fil, database, nettside)
Transform → Rens, valider, normaliser
Load      → Lagre til destinasjon (database, vektordatabase, datalager)
```

Som AI Engineer bygger du ETL-pipelines for å mate LLM-systemer med oppdaterte, rene data. Dårlig data = dårlige AI-svar.

**Verktøy vi bruker — alle gratis:**
- `requests` — HTTP-kall til APIer
- `pydantic` — Datavalidering med typer
- `sqlite3` — Enkel lokal database (innebygd i Python)
- `duckdb` — Analytisk SQL-database (pandas-erstatning)

In [ ]:
%pip install -q pydantic requests duckdb

---

## Steg 1: Extract — Hent data

In [ ]:
import requests
import json
from datetime import datetime

# Vi bruker en åpen, gratis API: JSONPlaceholder (simulerer dokumenter)
# I virkeligheten: SPK-API, SharePoint, database, PDF-parser osv.
URL = "https://jsonplaceholder.typicode.com/posts"

def hent_rådata(url: str, maks: int = 10) -> list[dict]:
    """Extract: Hent data fra API med feilhåndtering."""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()  # Kaster feil ved HTTP 4xx/5xx
        data = resp.json()[:maks]
        print(f"Hentet {len(data)} rader fra {url}")
        return data
    except requests.exceptions.RequestException as e:
        print(f"Feil ved henting: {e}")
        return []

rådata = hent_rådata(URL)
print(f"\nEksempel rad:\n{json.dumps(rådata[0], indent=2)}")

---

## Steg 2: Transform — Valider og rens

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError
from typing import Optional

class Dokument(BaseModel):
    """Pydantic-modell — definerer forventet datastruktur med validering."""
    id:        int
    tittel:    str
    innhold:   str
    kilde_id:  int
    hentet_kl: str = ""

    @field_validator("tittel", "innhold")
    @classmethod
    def ikke_tom(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("Feltet kan ikke være tomt")
        return v.strip()

    @field_validator("innhold")
    @classmethod
    def langt_nok(cls, v: str) -> str:
        if len(v) < 10:
            raise ValueError(f"Innhold for kort ({len(v)} tegn, minimum 10)")
        return v

def transformer(rad: dict) -> Optional[Dokument]:
    """Transform: Map og valider én rad."""
    try:
        return Dokument(
            id=rad["id"],
            tittel=rad.get("title", ""),
            innhold=rad.get("body", ""),
            kilde_id=rad.get("userId", 0),
            hentet_kl=datetime.now().isoformat(),
        )
    except ValidationError as e:
        print(f"  Valideringsfeil (id={rad.get('id')}): {e.errors()[0]['msg']}")
        return None

# Test med gyldige og ugyldige data
test_rader = [
    {"id": 1, "title": "Test",  "body": "Dette er nok innhold", "userId": 1},
    {"id": 2, "title": "",     "body": "Tomt tittel",           "userId": 1},
    {"id": 3, "title": "Kort", "body": "For kort",              "userId": 1},
]

print("Validering:")
for rad in test_rader:
    dok = transformer(rad)
    if dok:
        print(f"  ✅ id={dok.id}: '{dok.tittel[:30]}'")

In [ ]:
# Transformer alle rader fra API
dokumenter = [d for rad in rådata if (d := transformer(rad)) is not None]
print(f"\n{len(rådata)} rader hentet → {len(dokumenter)} gyldige dokumenter")

---

## Steg 3: Load — Lagre til database

In [ ]:
import duckdb

# DuckDB: lynrask analytisk database, lagrer til fil
con = duckdb.connect("pipeline.duckdb")

con.execute("""
    CREATE TABLE IF NOT EXISTS dokumenter (
        id        INTEGER PRIMARY KEY,
        tittel    VARCHAR,
        innhold   VARCHAR,
        kilde_id  INTEGER,
        hentet_kl VARCHAR,
        tegn_ant  INTEGER
    )
""")

# Slett eksisterende data (idempotent pipeline)
con.execute("DELETE FROM dokumenter")

# Last inn
for dok in dokumenter:
    con.execute(
        "INSERT INTO dokumenter VALUES (?, ?, ?, ?, ?, ?)",
        [dok.id, dok.tittel, dok.innhold, dok.kilde_id, dok.hentet_kl, len(dok.innhold)]
    )

# Verifiser
resultat = con.execute("SELECT COUNT(*), AVG(tegn_ant) FROM dokumenter").fetchone()
print(f"Lastet {resultat[0]} dokumenter, snitt {resultat[1]:.0f} tegn")

# Analytisk spørring — DuckDB er raskt på dette
print("\nDokumenter per kilde:")
for rad in con.execute("SELECT kilde_id, COUNT(*) FROM dokumenter GROUP BY kilde_id").fetchall():
    print(f"  Kilde {rad[0]}: {rad[1]} dokumenter")

---

## Del 4: Fullstendig pipeline-klasse

In [ ]:
from dataclasses import dataclass, field

@dataclass
class PipelineRapport:
    hentet:   int = 0
    gyldige:  int = 0
    lastet:   int = 0
    feil:     list = field(default_factory=list)

    def __str__(self):
        return (f"Hentet: {self.hentet} | Gyldige: {self.gyldige} | "
                f"Lastet: {self.lastet} | Feil: {len(self.feil)}")

def kjør_pipeline(url: str, db_sti: str = "pipeline.duckdb") -> PipelineRapport:
    rapport = PipelineRapport()

    # Extract
    rådata = hent_rådata(url, maks=20)
    rapport.hentet = len(rådata)

    # Transform
    dokumenter = []
    for rad in rådata:
        dok = transformer(rad)
        if dok:
            dokumenter.append(dok)
        else:
            rapport.feil.append(rad.get("id"))
    rapport.gyldige = len(dokumenter)

    # Load
    con = duckdb.connect(db_sti)
    con.execute("DELETE FROM dokumenter")
    for dok in dokumenter:
        con.execute(
            "INSERT INTO dokumenter VALUES (?, ?, ?, ?, ?, ?)",
            [dok.id, dok.tittel, dok.innhold, dok.kilde_id, dok.hentet_kl, len(dok.innhold)]
        )
    rapport.lastet = len(dokumenter)
    con.close()

    return rapport

rapport = kjør_pipeline(URL)
print(f"Pipeline ferdig: {rapport}")

---

## Oppsummering

| Steg | Verktøy | Nøkkelpoeng |
|------|---------|------------|
| Extract | `requests` | Alltid feilhåndtering + timeout |
| Transform | `pydantic` | Valider tidlig, feil koster lite |
| Load | `duckdb` | Idempotent (kjør samme pipeline flere ganger trygt) |

---

## Hva er neste steg?

**Neste: `15_dbt_and_snowflake.ipynb`** — dbt gjør SQL-transformasjoner testbare og dokumenterte. Du lærer modeller, tester og kjøremodus — med DuckDB lokalt (gratis).